# 13 — Physical + Residual Decomposition

**Context Demystifies Forecasting**

Notebook 07 showed the experimental distinction:

```text
output correction repairs forecasts
latent constraints shape forecasts
```

This notebook makes the central decomposition explicit:

```text
latent_state = physical_state + residual_state
```

The goal is to understand what is being constrained when constraints act inside hidden state rather than only on final predictions.


## Core idea

A forecast model can produce accurate predictions while its hidden state remains physically hard to interpret.

Phys-JEPA motivates a different design:

1. separate the latent state into a physical component and a residual component
2. constrain the physical component's transition
3. preserve residual flexibility for unmodeled dynamics

In this toy notebook, the physical component is a smooth structured process and the residual component is short-scale variation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")

## Build a toy system

The observable sequence is constructed as:

```text
observed = physical_state + residual_state
```

The physical state is smooth and constrained. The residual state is smaller, irregular, and not modeled as a clean physical transition.


In [ ]:
n = 900
t = np.linspace(0, 18 * np.pi, n)

physical_state = (
    1.0 * np.sin(t)
    + 0.35 * np.sin(0.5 * t + 0.7)
    + 0.010 * t
)

residual_state = (
    0.08 * rng.normal(size=n)
    + 0.04 * np.sin(3.7 * t + 0.4)
)

observed = physical_state + residual_state

df = pd.DataFrame({
    "t": t,
    "physical_state": physical_state,
    "residual_state": residual_state,
    "observed": observed,
})

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["t"], df["observed"], label="observed")
ax.plot(df["t"], df["physical_state"], label="physical state")

ax.set_title("Observed sequence and physical state")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_observed_and_physical.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["t"], df["physical_state"], label="physical state")
ax.plot(df["t"], df["residual_state"], label="residual state")

ax.set_title("Physical and residual components")
ax.set_xlabel("time")
ax.set_ylabel("component value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_latent_decomposition.png", dpi=160)
plt.show()

## Train / forecast split

We treat the first part of the sequence as observed history and forecast the remaining horizon.


In [ ]:
train_end = 620

history = df.iloc[:train_end].copy()
future = df.iloc[train_end:].copy()

history_t = history["t"].to_numpy()
future_t = future["t"].to_numpy()

history_y = history["observed"].to_numpy()
truth_y = future["observed"].to_numpy()
truth_physical = future["physical_state"].to_numpy()
truth_residual = future["residual_state"].to_numpy()

print(f"history length: {len(history)}")
print(f"future horizon: {len(future)}")

## Estimate physical state from history

To keep this notebook transparent, we use a small basis model to estimate the structured physical component:

```text
physical_state ≈ trend + sin(t) + cos(t) + sin(0.5t) + cos(0.5t)
```

This is a simple latent-state estimator, not a neural network.


In [ ]:
def design_matrix(x):
    return np.column_stack([
        np.ones_like(x),
        x,
        np.sin(x),
        np.cos(x),
        np.sin(0.5 * x),
        np.cos(0.5 * x),
    ])

X_history = design_matrix(history_t)
coef, *_ = np.linalg.lstsq(X_history, history_y, rcond=None)

def estimate_physical(x):
    return design_matrix(x) @ coef

estimated_history_physical = estimate_physical(history_t)
estimated_future_physical = estimate_physical(future_t)

history_residual_estimate = history_y - estimated_history_physical

coef

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(history_t, history_y, label="observed history")
ax.plot(history_t, estimated_history_physical, label="estimated physical state")
ax.plot(history_t, history_residual_estimate, label="estimated residual state")

ax.set_title("Estimated decomposition from history")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_estimated_history_decomposition.png", dpi=160)
plt.show()

## Residual forecast modes

We compare four rollouts:

1. **physical only**  
   Forecast only the constrained physical state.

2. **residual only**  
   Forecast only the recent residual average.

3. **physical + residual mean**  
   Add a residual correction to the physical forecast.

4. **physical + residual autoregression**  
   Add a small autoregressive residual forecast.

The point is to separate stable structure from flexible residue.


In [ ]:
def residual_mean_forecast(history_residual, horizon, window=40):
    return np.full(horizon, np.mean(history_residual[-window:]))

def residual_ar_forecast(history_residual, horizon, window=80, damping=0.92):
    r = np.asarray(history_residual[-window:])
    x = r[:-1]
    y = r[1:]

    if np.std(x) < 1e-12:
        phi = 0.0
    else:
        phi = float(np.dot(x, y) / np.dot(x, x))

    phi = float(np.clip(phi, -0.95, 0.95)) * damping

    forecast = np.zeros(horizon)
    forecast[0] = phi * history_residual[-1]

    for i in range(1, horizon):
        forecast[i] = phi * forecast[i - 1]

    return forecast, phi

horizon = len(future_t)

physical_only = estimated_future_physical
residual_only = residual_mean_forecast(history_residual_estimate, horizon)
physical_plus_residual_mean = physical_only + residual_only
residual_ar, residual_phi = residual_ar_forecast(history_residual_estimate, horizon)
physical_plus_residual_ar = physical_only + residual_ar

residual_phi

In [ ]:
rollout_df = pd.DataFrame({
    "t": future_t,
    "truth": truth_y,
    "truth_physical": truth_physical,
    "truth_residual": truth_residual,
    "physical_only": physical_only,
    "residual_only": residual_only,
    "physical_plus_residual_mean": physical_plus_residual_mean,
    "physical_plus_residual_ar": physical_plus_residual_ar,
})

rollout_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(rollout_df["t"], rollout_df["truth"], label="truth")
ax.plot(rollout_df["t"], rollout_df["physical_only"], label="physical only")
ax.plot(rollout_df["t"], rollout_df["physical_plus_residual_mean"], label="physical + residual mean")
ax.plot(rollout_df["t"], rollout_df["physical_plus_residual_ar"], label="physical + residual AR")

ax.set_title("Physical/residual rollout comparison")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_physical_residual_rollout.png", dpi=160)
plt.show()

## Metrics

We measure full-horizon MAE and RMSE.

The expected pattern:

```text
physical only
  stable but misses residual variation

residual only
  incomplete

physical + residual
  best reconstruction when residual is useful
```


In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

methods = [
    "physical_only",
    "residual_only",
    "physical_plus_residual_mean",
    "physical_plus_residual_ar",
]

metrics = pd.DataFrame([
    {
        "method": method,
        "MAE": mae(rollout_df["truth"], rollout_df[method]),
        "RMSE": rmse(rollout_df["truth"], rollout_df[method]),
    }
    for method in methods
])

best_rmse = metrics["RMSE"].min()
metrics["RMSE_vs_best"] = metrics["RMSE"] / best_rmse

metrics.to_csv(RESULTS / "13_decomposition_metrics.csv", index=False)
rollout_df.to_csv(RESULTS / "13_rollouts.csv", index=False)

metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(metrics))
ax.bar(x, metrics["RMSE"])
ax.set_xticks(x)
ax.set_xticklabels(metrics["method"], rotation=25, ha="right")

ax.set_title("Decomposition forecast error")
ax.set_ylabel("RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "13_decomposition_error.png", dpi=160)
plt.show()

## Component error

A physically meaningful latent decomposition should allow the physical component to be evaluated separately.

This is the interpretability advantage: the model can be wrong in a component-specific way.


In [ ]:
component_metrics = pd.DataFrame([
    {
        "component": "physical_state",
        "MAE": mae(truth_physical, estimated_future_physical),
        "RMSE": rmse(truth_physical, estimated_future_physical),
    },
    {
        "component": "residual_state_AR",
        "MAE": mae(truth_residual, residual_ar),
        "RMSE": rmse(truth_residual, residual_ar),
    },
    {
        "component": "residual_state_mean",
        "MAE": mae(truth_residual, residual_only),
        "RMSE": rmse(truth_residual, residual_only),
    },
])

component_metrics.to_csv(RESULTS / "13_component_metrics.csv", index=False)
component_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(future_t, truth_physical, label="true physical state")
ax.plot(future_t, estimated_future_physical, label="estimated physical state")
ax.plot(future_t, truth_residual, label="true residual state")
ax.plot(future_t, residual_ar, label="estimated residual AR")

ax.set_title("Component-level interpretability")
ax.set_xlabel("time")
ax.set_ylabel("component value")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_component_interpretability.png", dpi=160)
plt.show()

## Horizon stability

The physical component should dominate long-horizon stability.

We compare cumulative RMSE over time for each rollout.


In [ ]:
def cumulative_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    sq_error = (y_true - y_pred) ** 2
    return np.sqrt(np.cumsum(sq_error) / np.arange(1, len(sq_error) + 1))

horizon_df = pd.DataFrame({
    "horizon": np.arange(1, horizon + 1)
})

for method in methods:
    horizon_df[method] = cumulative_rmse(rollout_df["truth"], rollout_df[method])

horizon_df.to_csv(RESULTS / "13_horizon_stability.csv", index=False)

horizon_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for method in methods:
    ax.plot(horizon_df["horizon"], horizon_df[method], label=method)

ax.set_title("Horizon stability by decomposition mode")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("cumulative RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "13_horizon_stability.png", dpi=160)
plt.show()

## Interpretation

This notebook separates two questions:

```text
Can the model forecast?
Can the model explain what part of state is being forecast?
```

The physical component carries stable structure.

The residual component carries variation not explained by that structure.

That decomposition makes the latent state more interpretable:

```text
latent_state = physical_state + residual_state
```

For Phys-JEPA-style forecasting, the engineering lesson is:

> constrain the part of latent state that corresponds to measurable structure, while preserving residual capacity for what is not yet modeled.


## Next

Notebook 17 should move this from toy signals to a real dataset.

Candidate:

```text
Jena Climate
```

The goal:

> test whether physical/residual decomposition still helps when the system is real and the constraints are imperfect.
